In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Load dataset
df = pd.read_csv("hab_ndbc_merged.csv")

# Convert dates
df["week_start"] = pd.to_datetime(df["week_start"])
df["sample_date"] = pd.to_datetime(df["sample_date"])

# Create date-based features
df["weekofyear"] = df["week_start"].dt.isocalendar().week.astype(int)
df["dayofyear"] = df["week_start"].dt.dayofyear

# Target variable
target = "potential_bloom"

# Features
features = [
    "station",
    "latitude",
    "longitude",
    "month",
    "year",
    "temp",
    "silicate",
    "nitrate",
    "avg_chloro",
    "sst_roll_14d",
    "anom_roll_14d",
    "sst_roc_3d",
    "warm_degree_days_14d",
    "above_avg",
    "pda",
    "weekofyear",
    "dayofyear"
]

X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Separate categorical and numeric columns
categorical_features = ["station"]
numeric_features = [col for col in features if col not in categorical_features]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

# Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    min_samples_leaf=2,
    n_jobs=-1
)

# Pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", rf_model)
    ]
)

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

For this model, I used a Random Forest classification model to predict whether a location and time period had a potential algal bloom. The target variable was `potential_bloom`, which indicates whether bloom conditions were present. The model used environmental, biological, spatial, and time-based features, including sea surface temperature trends, chlorophyll, nitrate, silicate, location, month, year, and PDA.

A Random Forest model works by building many individual decision trees and combining their predictions. This makes the model more stable than a single decision tree because it reduces the chance of relying too heavily on one specific pattern in the data. Since algal blooms are influenced by many interacting environmental factors, Random Forest is a good choice because it can capture nonlinear relationships and feature interactions.

I also used `class_weight="balanced"` because potential bloom cases were less common than non-bloom cases in the dataset. This helps the model pay more attention to the minority class instead of mostly predicting the majority class.

In [ ]:
# Get station one-hot encoded feature names
encoded_station_names = model.named_steps["preprocessor"] \
    .named_transformers_["cat"] \
    .get_feature_names_out(["station"])

feature_names = list(encoded_station_names) + numeric_features

# Random Forest feature importances
importances = model.named_steps["classifier"].feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feature_importance_df.head(15)

The feature importance results show which variables the Random Forest model relied on most when predicting potential algal blooms. Features with higher importance values contributed more to the model’s decisions.

PDA was included as a key feature because it is directly connected to algal bloom activity. If PDA appears near the top of the feature importance chart, that suggests the model found it highly useful for identifying bloom conditions. This makes sense because PDA can act as a biological indicator of harmful algal bloom presence.

Other important features may include variables such as `avg_chloro`, `sst_roll_14d`, `anom_roll_14d`, and `warm_degree_days_14d`. These features are meaningful because algal blooms are often related to changes in water temperature, chlorophyll levels, and sustained warm conditions. For example, high chlorophyll can indicate more phytoplankton activity, while warmer sea surface temperatures may create conditions that support bloom growth.

Location and time-based features, such as `station`, `latitude`, `longitude`, `month`, and `dayofyear`, can also be important because bloom risk is not the same everywhere or during every season. Some stations may experience bloom conditions more often due to local environmental patterns, and blooms may be more likely during certain parts of the year.

Overall, the feature importance chart helps show that the model is using a combination of biological indicators, ocean temperature patterns, nutrient conditions, and seasonal/spatial information to predict potential algal blooms. Since PDA is closely related to bloom identification, its importance should be interpreted carefully, but it is still useful for understanding bloom risk in this project.